# Week 2: Sampling Configuration Experiment

This notebook supports the Week 2 discussion assignment by running the **same prompt through the same language model three times** while changing only the requested sampling configuration.

The three configurations are:

1. Temperature `0`
2. Temperature `0.7` with top-p `0.9`
3. Temperature `1.2` with top-k `50`

The generated outputs should be left exactly as produced. Do not edit or clean up model responses before comparing them.


## Prompt and task

**Prompt**

> Explain how a large language model generates the next token in a response. Write the explanation for a non-technical audience in no more than 100 words.

**Task**

The prompt asks the model to explain next-token generation accurately, concisely, and in language understandable to a non-technical audience. This makes it possible to compare factual accuracy, style, length, and clarity as sampling becomes more or less random.


## Setup

This notebook uses `Qwen/Qwen2.5-1.5B-Instruct` through Hugging Face Transformers because Transformers exposes temperature, top-p, and top-k directly.

If you are running in Google Colab, uncomment the install line below on the first run. No API key is required.


In [ ]:
# In Google Colab, uncomment this line on the first run:
# !pip install -q transformers torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)
print("Model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()


## Prepare the identical prompt

The prompt is defined once and reused for all three runs so the sampling configuration is the experimental variable.


In [ ]:
PROMPT = (
    "Explain how a large language model generates the next token in a response. "
    "Write the explanation for a non-technical audience in no more than 100 words."
)

messages = [
    {"role": "user", "content": PROMPT}
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer(
    [formatted_prompt],
    return_tensors="pt",
).to(device)

print(PROMPT)


## Generation helper

The helper below returns only the model's newly generated response, not the prompt. Each experiment cell prints the response exactly as generated and also stores it in a variable for later comparison.


In [ ]:
def generate_response(**sampling_settings):
    with torch.no_grad():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=160,
            pad_token_id=tokenizer.eos_token_id,
            **sampling_settings,
        )

    new_tokens = generated[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    print(response)
    return response


## Run 1: Temperature 0

With sampling disabled, generation chooses the highest-probability next token at each step. This is the practical equivalent of temperature 0 for this experiment.


In [ ]:
output_temperature_0 = generate_response(
    do_sample=False,
)


## Run 2: Temperature 0.7 with top-p 0.9

This run samples from tokens inside the smallest probability mass totaling 0.9. `top_k=0` disables top-k filtering so top-p is the requested truncation method.


In [ ]:
output_temperature_07_top_p_09 = generate_response(
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    top_k=0,
)


## Run 3: Temperature 1.2 with top-k 50

This run increases randomness and limits candidate selection to the 50 highest-probability tokens. `top_p=1.0` leaves nucleus filtering effectively unrestricted so top-k is the requested truncation method.


In [ ]:
output_temperature_12_top_k_50 = generate_response(
    do_sample=True,
    temperature=1.2,
    top_p=1.0,
    top_k=50,
)


## Preserve and compare the outputs

**Important:** Runs 2 and 3 use stochastic sampling. If you rerun those cells, their responses may change. Once you have the outputs you want to analyze, save the notebook so the exact generated text remains recorded in the cell outputs.


In [ ]:
results = {
    "Temperature 0": output_temperature_0,
    "Temperature 0.7, top-p 0.9": output_temperature_07_top_p_09,
    "Temperature 1.2, top-k 50": output_temperature_12_top_k_50,
}

for configuration, response in results.items():
    word_count = len(response.split())
    print(f"{configuration}: {word_count} words")


## Analysis notes

After running all three configurations, make your own observations before drafting the 200–300 word discussion post.

Address these questions:

- How did the outputs differ in **style**?
- How did they differ in **length**?
- Did any response become less accurate, less clear, or fail the 100-word constraint?
- Which configuration was best for this explanatory task, and why?
- Which inference stage produced these differences?
- How would you explain the weakest response to a non-technical stakeholder?

### Inference concept to connect to your observations

The differences primarily arise during **next-token sampling/selection**. The model first produces logits and a probability distribution over possible next tokens. Temperature changes how sharp or flat that distribution is, while top-p and top-k restrict the candidate tokens that remain eligible for sampling.

Write the final discussion post only after reviewing the actual outputs produced above.
